In [22]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv  
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

In [23]:
load_dotenv()

True

In [24]:
# This is the global variable to store document content
document_content = ""

In [25]:

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [26]:
@tool
def update(content: str) -> str:
    """Updates the document with the provided content."""
    global document_content
    document_content = content
    return f"Document has been updated successfully! The current content is:\n{document_content}"


@tool
def save(filename: str) -> str:
    """Save the current document to a text file and finish the process.
    
    Args:
        filename: Name for the text file.
    """

    global document_content

    if not filename.endswith('.txt'):
        filename = f"{filename}.txt"


    try:
        with open(filename, 'w') as file:
            file.write(document_content)
        print(f"\n💾 Document has been saved to: {filename}")
        return f"Document has been saved successfully to '{filename}'."
    
    except Exception as e:
        return f"Error saving document: {str(e)}"
    

tools = [update, save]
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
).bind_tools(tools)

In [27]:
def our_agent(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content=f"""
    You are Drafter, a helpful writing assistant. You are going to help the user update and modify documents.
    
    - If the user wants to update or modify content, use the 'update' tool with the complete updated content.
    - If the user wants to save and finish, you need to use the 'save' tool.
    - Make sure to always show the current document state after modifications.
    
    The current document content is:{document_content}
    """)

    if not state["messages"]:
        user_input = "I'm ready to help you update a document. What would you like to create?"
        user_message = HumanMessage(content=user_input)

    else:
        user_input = input("\nWhat would you like to do with the document? ")
        print(f"\n👤 USER: {user_input}")
        user_message = HumanMessage(content=user_input)

    all_messages = [system_prompt] + list(state["messages"]) + [user_message]

    response = model.invoke(all_messages)

    print(f"\n🤖 AI: {response.content}")
    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"🔧 USING TOOLS: {[tc['name'] for tc in response.tool_calls]}")

    return {"messages": list(state["messages"]) + [user_message, response]}


In [28]:
def should_continue(state: AgentState) -> str:
    """Determine if we should continue or end the conversation."""

    messages = state["messages"]
    
    if not messages:
        return "continue"
    
    # This looks for the most recent tool message....
    for message in reversed(messages):
        # ... and checks if this is a ToolMessage resulting from save
        if (isinstance(message, ToolMessage) and 
            "saved" in message.content.lower() and
            "document" in message.content.lower()):
            return "end" # goes to the end edge which leads to the endpoint
        
    return "continue"

In [29]:
def print_messages(messages):
    """Function I made to print the messages in a more readable format"""
    if not messages:
        return
    
    for message in messages[-3:]:
        if isinstance(message, ToolMessage):
            print(f"\n🛠️ TOOL RESULT: {message.content}")

In [30]:
graph = StateGraph(AgentState)

graph.add_node("agent", our_agent)
graph.add_node("tools", ToolNode(tools))

graph.set_entry_point("agent")

graph.add_edge("agent", "tools")


graph.add_conditional_edges(
    "tools",
    should_continue,
    {
        "continue": "agent",
        "end": END,
    },
)

app = graph.compile()

In [31]:
def run_document_agent():
    print("\n ===== DRAFTER =====")
    
    state = {"messages": []}
    
    for step in app.stream(state, stream_mode="values"):
        if "messages" in step:
            print_messages(step["messages"])
    
    print("\n ===== DRAFTER FINISHED =====")

if __name__ == "__main__":
    run_document_agent()


 ===== DRAFTER =====

🤖 AI: 
🔧 USING TOOLS: ['update']

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document.



What would you like to do with the document?  write me an email



👤 USER: write me an email

🤖 AI: 
🔧 USING TOOLS: ['update']

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document.

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document. 

Subject: Test Email
Dear Recipient,
This is a test email.
Best regards,
Sender



What would you like to do with the document?  you didnt ask about what



👤 USER: you didnt ask about what

🤖 AI: Let's start again. What kind of email would you like to write? Is it a formal or informal email? Who is the recipient? What is the purpose of the email? 

(No changes will be made to the document until you provide more information)

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document. 

Subject: Test Email
Dear Recipient,
This is a test email.
Best regards,
Sender

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document. 

Subject: Test Email
Dear Recipient,
This is a test email.
Best regards,
Sender


...


👤 USER: do you know todays date?

🤖 AI: I'm not aware of the current date. I'm a text-based AI assistant and do not have real-time access to the current date or time. However, I can help you draft an email with a specific deadline if you provide me with the current date. 

Would you like to provide the current date so I can help you with the deadline calculation?

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document. 

Subject: Reminder: Nakuja Project Submission Deadlines
Dear Student,
This is a reminder that the submission deadlines for the Nakuja project are approaching. The deadlines are two weeks from now, at 11:59 PM. Please make sure to submit your work on time.
Best regards,
[Your Name]

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document. 

Subject: Reminder: Nakuja Project Submission Deadlines
Dear Student,
This is a reminder that the submission deadlines for the Nakuja proje


What would you like to do with the document?  today is july 6 so rewrite with exact date two weeks from now



👤 USER: today is july 6 so rewrite with exact date two weeks from now

🤖 AI: 
🔧 USING TOOLS: ['update']

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document. 

Subject: Reminder: Nakuja Project Submission Deadlines
Dear Student,
This is a reminder that the submission deadlines for the Nakuja project are approaching. The deadlines are on July 20th, at 11:59 PM. Please make sure to submit your work on time.
Best regards,
[Your Name]



What would you like to do with the document?  save



👤 USER: save

🤖 AI: 
🔧 USING TOOLS: ['save']

🛠️ TOOL RESULT: Document has been updated successfully! The current content is:
This is a new document. 

Subject: Reminder: Nakuja Project Submission Deadlines
Dear Student,
This is a reminder that the submission deadlines for the Nakuja project are approaching. The deadlines are on July 20th, at 11:59 PM. Please make sure to submit your work on time.
Best regards,
[Your Name]

💾 Document has been saved to: NakujaProjectReminder.txt

🛠️ TOOL RESULT: Document has been saved successfully to 'NakujaProjectReminder.txt'.

 ===== DRAFTER FINISHED =====
